# Memory: generators of `DetectorData` and pixel input

For one year of LiteBIRD a process can hold billions of samples, so what
pysanepic keeps per sample matters:

* the TODs are used without copies, and P, N⁻¹ and Pᵀ are applied one chunk at
  a time;
* per sample pysanepic keeps only the pixel (int32) and γ cos 2a, γ sin 2a
  (float32): 12 bytes, 16 during the setup;
* `make_maps` accepts a **generator** of `DetectorData`: the pointings
  (theta, phi, psi in float64, 24 bytes per sample) of only one detector exist
  at a time;
* a `DetectorData` can carry the **pixel index** `pix` (with its `nside`)
  instead of theta and phi, e.g. when the pixels come from another pipeline or
  from disk.

This notebook uses a synthetic scan and does not need litebird_sim; the
litebird_sim interface (`Simulation.make_sanepic_gls_map`) already passes the
detectors with a generator. No HWP.

In [ ]:
import time
import tracemalloc

import healpy as hp
import numpy as np
from pysanepic import DetectorData, make_maps

In [ ]:
nside, fs, n = 64, 10.0, 2_000_000  # 2.3 days at 10 Hz per detector
pol_angles = [0.0, np.pi / 4, np.pi / 8, 3 * np.pi / 8]
rng = np.random.default_rng(0)
sky = rng.normal(0, 1e-4, (3, 12 * nside**2))  # Ecliptic coordinates


def scan(n, offset):
    # spinning scan: the spin axis goes around the ecliptic in n samples
    t = np.arange(n)
    lam = 2 * np.pi * t / n + offset
    spin = 2 * np.pi * t / 3000.0
    a = np.array([np.cos(lam), np.sin(lam), np.zeros(n)])
    z = np.array([np.zeros(n), np.zeros(n), np.ones(n)])
    b = np.cross(z, a, axis=0)
    beta = np.radians(50.0)
    v = np.cos(beta) * a + np.sin(beta) * (np.cos(spin) * z + np.sin(spin) * b)
    theta, phi = hp.vec2ang(v.T)
    psi = spin % (2 * np.pi)
    return theta, phi, psi


def detectors(use_pix=False):
    # one detector at a time: its pointings are freed when the next one is made
    for k, pol_angle in enumerate(pol_angles):
        theta, phi, psi = scan(n, offset=k)
        pix = hp.ang2pix(nside, theta, phi)
        a = psi + pol_angle
        tod = sky[0, pix] + sky[1, pix] * np.cos(2 * a) + sky[2, pix] * np.sin(2 * a)
        tod = (tod + rng.normal(0, 1e-4, n)).astype(np.float32)
        pointing = dict(pix=pix, nside=nside) if use_pix else dict(theta=theta, phi=phi)
        yield DetectorData(
            tod=tod, psi=psi, coordinates="E", **pointing,
            sampling_rate_hz=fs, net_ukrts=40.0, fknee_hz=0.02, alpha=1.0, fmin_hz=1e-5,
            pol_angle_rad=pol_angle,
        )

## Memory used by `make_maps`

The TODs are made before the measurement, so the peak counts only what
pysanepic allocates (with a list of `DetectorData`, the pointings of all the
detectors would also be in memory).

In [ ]:
data = list(detectors())
tods = [d.tod for d in data]
nsamples = sum(t.size for t in tods)

tracemalloc.start()
t0 = time.time()
res = make_maps(data, nside, coordinates="E", nthreads=4)
peak = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()
print(f"{res.iterations} iterations in {time.time() - t0:.1f} s, converged: {res.converged}")
print(f"{nsamples:.2e} samples; TODs {sum(t.nbytes for t in tods) / 1e6:.0f} MB; "
      f"peak allocated by make_maps {peak / 1e6:.0f} MB = {peak / nsamples:.1f} bytes per sample")

Besides the 16 bytes per sample of the setup, the peak includes a few maps
(3 × 49152 pixels here) and the temporaries of one detector (pixels and angles
in float64 while they are computed): here each detector is a quarter of the
data, so they weigh; with many detectors, or one `DetectorData` per day as in
litebird_sim, the peak approaches 16 bytes per sample.

## Pixel input

The same maps from pixel indices instead of theta and phi: `pix` must be at the
`nside` and in the `coordinates` of the output maps, and `psi` in the same
coordinates.

In [ ]:
rng = np.random.default_rng(0)
sky = rng.normal(0, 1e-4, (3, 12 * nside**2))
ref = make_maps(detectors(), nside, coordinates="E", nthreads=4)
rng = np.random.default_rng(0)
sky = rng.normal(0, 1e-4, (3, 12 * nside**2))
res = make_maps(detectors(use_pix=True), nside, coordinates="E", nthreads=4)
print("same maps:", np.array_equal(res.maps, ref.maps))

## On a cluster

On G100 (CINECA), 16 detectors of the LiteBIRD M2_L channel at 75 Hz for 7 days
(7.3·10⁸ samples), nside 512, 16 MPI processes × 3 threads, simulated with
litebird_sim: the peak memory per process went from 6.5 GB to 2.7 GB (1.2 GB
are the simulation), and the GLS time from 190 s to 120 s (ducc0 FFTs, which
parallelize a single 1D transform). With one detector per process, a year at
75 Hz needs about 30–40 GB per process for pysanepic, plus the TOD.